In [ ]:
import pandas as pd, numpy as np

In [ ]:
location = "india"
directory = "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/"
results_dir = (
    "../results"
)

### WRA

In [ ]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA"),
    columns=wra_columns.keys(),
)

In [ ]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [ ]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [ ]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [ ]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Adult mortality

Adult mortality included in HH for India (recent household members who died).

In [ ]:
%%time

household_columns = {
    "hv005": "weight",
    "sh70": "any_died",
    "sh71": "num_died",
    "hv270": "wealth_quintile",
}
MAX_NUM_DEATHS = 5
death_columns = {
    "sh73": "sex",
    "sh74u": "age_at_death_unit",
    "sh74n": "age_at_death",
    "sh75m": "month_of_death",
    "sh75y": "year_of_death",
    "sh76": "death_violence_or_accident",
    "sh77": "death_during_pregnancy_or_childbirth",
}
columns = list(household_columns.keys())
for death_num in range(1, MAX_NUM_DEATHS + 1):
    columns += [c + "_" + str(death_num) for c in death_columns.keys()]

adult_mortality_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA", columns=columns
)
adult_mortality_data

In [ ]:
# inspired by https://stackoverflow.com/a/67393747/
adult_mortality_data_reshaped = adult_mortality_data[
    [c for c in adult_mortality_data.columns if c.split("_")[0] in death_columns.keys()]
].copy()
adult_mortality_data_reshaped.columns = adult_mortality_data_reshaped.columns.str.split(
    "_", expand=True
)
adult_mortality_data_reshaped

In [ ]:
adult_mortality_data_reshaped[list(household_columns.keys())] = adult_mortality_data[
    list(household_columns.keys())
]
adult_mortality_data_reshaped

In [ ]:
# Get a row per death
adult_mortality_data_reshaped = (
    adult_mortality_data_reshaped.set_index(list(household_columns.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(household_columns)}"])
)
adult_mortality_data_reshaped

In [ ]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [ ]:
adult_mortality_data = (
    adult_mortality_data_reshaped[
        list(household_columns.keys()) + list(death_columns.keys())
    ]
    .rename(columns=household_columns)
    .rename(columns=death_columns)
)
adult_mortality_data["wealth_quintile"] = recode_wealth_quintile(
    adult_mortality_data.wealth_quintile
)
adult_mortality_data["weight"] = adult_mortality_data.weight / 1_000_000
adult_mortality_data

### Births

In [ ]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA"),
    columns=birth_columns.keys(),
)
birth_data

In [ ]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

### Household members

In [ ]:
%%time

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw_adult",
    "ha56": "hemoglobin_adjusted_adult",
    "ha57": "anemia_adult",
    "hc53": "hemoglobin_raw_child",
    "hc56": "hemoglobin_adjusted_child",
    "hc57": "anemia_child", 
}
hhm_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA"),
    columns=hhm_columns.keys(),
)
hhm_data

In [ ]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [ ]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [ ]:
hhm_data["sex"] = hhm_data.sex.str.title()

In [ ]:
# Interesting -- sometimes age is quite off between hemoglobin and base.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

In [ ]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

In [ ]:
hhm_data["pregnant"] = hhm_data.currently_pregnant.map({"not pregnant, don't know": "not_pregnant", "pregnant": "pregnant"})
hhm_data.loc[hhm_data.sex != 'Female', "pregnant"] = "not_pregnant"
hhm_data.loc[(hhm_data.age < 15) | (hhm_data.age >= 50), "pregnant"] = "not_pregnant"
age_bin_edges = [0, 5, 15, 30, 50, 100]
age_group = pd.IntervalIndex(
    pd.cut(hhm_data.age, age_bin_edges, right=False, include_lowest=True)
)
hhm_data["age_start"] = age_group.left
hhm_data["age_end"] = age_group.right

In [ ]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [ ]:
for type in ["child", "adult"]:
    for base_col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
        col = f"{base_col}_{type}"
        hhm_data[col] = (
            hhm_data[col]
            .astype(str)
            .replace(
                {
                    "not tested": np.nan,
                    "not present": np.nan,
                    "refused": np.nan,
                    "other": np.nan,
                }
            )
            .astype(float)
        )

In [ ]:
for base_col in ["hemoglobin_raw", "hemoglobin_adjusted", "anemia"]:
    assert (hhm_data.filter(like=base_col).notnull().sum(axis=1) <= 1).all()
    hhm_data[base_col] = np.nan
    # Could use bfill instead of this loop, but it was incredibly slow for me
    for col in hhm_data.filter(like=base_col).columns:
        hhm_data[base_col] = hhm_data[base_col].fillna(hhm_data[col])

In [ ]:
assert (hhm_data[(hhm_data.sex == "male") & (hhm_data.age > 5)].hemoglobin_raw.isnull().all())

### Maternal mortality ratio

#### Maternal mortality rate

Not reported anywhere for India DHS, so we need to be extra careful since we can't cross-check.

In [ ]:
adult_mortality_data.death_during_pregnancy_or_childbirth.value_counts()

In [ ]:
adult_mortality_data.death_violence_or_accident.value_counts()

In [ ]:
adult_mortality_data["age_at_death_years"] = (
    adult_mortality_data.age_at_death_unit.map(
        {"year": 1, "months": 1 / 12, "days": 1 / 365.25}
    )
    * adult_mortality_data.age_at_death
)

In [ ]:
adult_mortality_data["adult_death"] = (
    adult_mortality_data.age_at_death_years >= 15
) & (adult_mortality_data.age_at_death_years < 50)

In [ ]:
adult_mortality_data["date_of_death"] = adult_mortality_data.year_of_death.replace(
    {"don't know": np.nan}
).astype(float) + (
    adult_mortality_data.month_of_death.map(
        {
            "january": 1,
            "february": 2,
            "march": 3,
            "april": 4,
            "may": 5,
            "june": 6,
            "july": 7,
            "august": 8,
            "september": 9,
            "october": 10,
            "november": 11,
            "december": 12,
        }
    )
    - 0.5
) * (
    1 / 12
)

In [ ]:
adult_mortality_data["date_of_birth"] = (
    adult_mortality_data.date_of_death - adult_mortality_data.age_at_death_years
)
adult_mortality_data["exposure_start"] = np.maximum(
    adult_mortality_data.date_of_birth + 15, 2011.0
)
adult_mortality_data["exposure_end"] = np.minimum(
    adult_mortality_data.date_of_birth + 50, adult_mortality_data.date_of_death
)
adult_mortality_data["exposure"] = np.maximum(
    adult_mortality_data.exposure_end - adult_mortality_data.exposure_start, 0
)
adult_mortality_data["weighted_exposure"] = (
    adult_mortality_data.weight * adult_mortality_data.exposure
)
adult_mortality_data

In [ ]:
living_exposure_data = hhm_data.copy()
living_exposure_data["date_of_birth"] = (
    (living_exposure_data.date_of_interview / 12) + 1900 - living_exposure_data.age
)
living_exposure_data["exposure_start"] = np.maximum(
    living_exposure_data.date_of_birth + 15, 2011.0
)
living_exposure_data["exposure_end"] = np.minimum(
    living_exposure_data.date_of_birth + 50,
    (living_exposure_data.date_of_interview / 12) + 1900,
)
living_exposure_data["exposure"] = np.maximum(
    living_exposure_data.exposure_end - living_exposure_data.exposure_start, 0
)
living_exposure_data["weighted_exposure"] = (
    living_exposure_data.weight * living_exposure_data.exposure
)
living_exposure_data

In [ ]:
(adult_mortality_data.adult_death * adult_mortality_data.weight).sum()

In [ ]:
adult_mortality_data.weighted_exposure.sum()

In [ ]:
((adult_mortality_data.adult_death * adult_mortality_data.weight).sum()) / (
    adult_mortality_data.weighted_exposure.sum()
    + living_exposure_data.weighted_exposure.sum()
)

In [ ]:
adult_mortality_data

In [ ]:
adult_mortality_data["maternal_death"] = (
    adult_mortality_data.adult_death
    & (adult_mortality_data.death_during_pregnancy_or_childbirth == "yes")
    & (adult_mortality_data.death_violence_or_accident != "yes")
)

In [ ]:
(adult_mortality_data.maternal_death * adult_mortality_data.weight).sum()

In [ ]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        df.weighted_exposure
    ).sum()

In [ ]:
maternal_mortality_data = pd.concat(
    [
        adult_mortality_data[adult_mortality_data.sex == "female"][
            ["wealth_quintile", "maternal_death", "weight", "weighted_exposure"]
        ],
        living_exposure_data[living_exposure_data.sex == "female"][
            ["wealth_quintile", "weight", "weighted_exposure"]
        ].assign(maternal_death=False),
    ],
    ignore_index=True,
)

In [ ]:
maternal_mortality_rate(maternal_mortality_data)

In [ ]:
maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

#### General fertility rate

In [ ]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [ ]:
fertility_event_data.weighted_birth_in_period.sum()

In [ ]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

In [ ]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [ ]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

In [ ]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

In [ ]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/india.csv",
    index=False,
)